# Modern Python: Pydantic and Requests
This notebook covers two essential tools for modern Python development:
1. **Pydantic**: Data validation and settings management using Python type annotations.
2. **Requests**: The gold standard for making HTTP requests to interact with APIs.

## 1. Pydantic: Data Validation
Pydantic ensures that the data your program receives matches the types you expect. If the data is wrong, it raises a clear error.

In [5]:
from pydantic import BaseModel, ValidationError
from typing import List, Optional

class User(BaseModel):
    id: int
    username: str
    email: str
    is_active: bool = True
    tags: List[str] = []

# Valid Data
try:
    user = User(id=1, username="biju_dev", email="test@example.com", tags=["python", "fastapi"])
    print("Successfully created user object:")
    print(user.model_dump()) # Converts object to dictionary
except ValidationError as e:
    print(e.json())

print("\n--- Testing Validation Error ---")
# Invalid Data (id should be int, but we pass a string that can't be converted)
try:
    bad_user = User(id="not-an-int", username="fail", email="wrong-email")
except ValidationError as e:
    print("Validation failed as expected:")
    print(e)

Successfully created user object:
{'id': 1, 'username': 'biju_dev', 'email': 'test@example.com', 'is_active': True, 'tags': ['python', 'fastapi']}

--- Testing Validation Error ---
Validation failed as expected:
1 validation error for User
id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not-an-int', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


## 2. Requests: Interacting with APIs
The `requests` library allows you to send HTTP requests (GET, POST, etc.) to web servers easily.

In [2]:
import requests

# Making a GET request to a sample API
response = requests.get("https://jsonplaceholder.typicode.com/posts/1")

if response.status_code == 200:
    data = response.json()
    print("API Response Data:")
    print(f"Title: {data['title']}")
else:
    print(f"Failed to fetch data. Status code: {response.status_code}")

# Making a POST request (Sending data)
payload = {"title": "foo", "body": "bar", "userId": 1}
post_response = requests.post("https://jsonplaceholder.typicode.com/posts", json=payload)
print(f"\nPOST Status Code: {post_response.status_code}")
print("POST Response:", post_response.json())

API Response Data:
Title: sunt aut facere repellat provident occaecati excepturi optio reprehenderit

POST Status Code: 201
POST Response: {'title': 'foo', 'body': 'bar', 'userId': 1, 'id': 101}


## 3. Combining Both: Fetching API Data into a Pydantic Model
This is a very common pattern in professional development.

In [6]:
class Post(BaseModel):
    userId: int
    id: int
    title: str
    body: str

response = requests.get("https://jsonplaceholder.typicode.com/posts/2")
if response.status_code == 200:
    # Validate the API response using our Pydantic model
    post_obj = Post(**response.json())
    print("Validated Post Object:")
    print(f"Post ID {post_obj.id} by User {post_obj.userId}: {post_obj.title}")

Validated Post Object:
Post ID 2 by User 1: qui est esse
